# STAR: Factory monitoring

Code for the factory-monitoring results of
**"STAR: Semantic-Criticality Tiered Allocation of Radio Resources via Zero-Shot Learning"**.

| Section | Paper |
|---|---|
| 2. Synthetic factory dataset | Sec. IV, Fig. 1, Table I |
| 3. SCI estimation via zero-shot learning | Algorithm 1 |
| 4. Tiered transmission and reception | Sec. III-B |
| 5. Speaker + judge evaluation | Sec. IV-A |
| 6. STAR vs conventional QAM | Fig. 4, Fig. 5 |
| 7. Resource elements per image | Fig. 6a |
| 8. Selective criticality | Fig. 6b |
| 9. SCI overlays and received images | Fig. 2, Fig. 3 |
| 10. Computation time | Sec. IV-C |

Run the notebook from the **repository root**. Results are written to `results/factory/` and figures to
`results/figures/factory/`. Sections 6 and 8 run the full $E_s/N_0$ sweep and take many hours.

## 1. Setup

In [ ]:
import os, sys, gc, time, random, pickle
import numpy as np, torch, requests
import matplotlib.pyplot as plt
from PIL import Image as PILImage

REPO_ROOT   = os.path.abspath(".")          # run from the repository root
RESULTS_DIR = os.path.join(REPO_ROOT, "results", "factory")
FIG_DIR     = os.path.join(REPO_ROOT, "results", "figures", "factory")
os.makedirs(RESULTS_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
sys.path.insert(0, os.path.join(REPO_ROOT, "factory_dataset"))

Model manager: the attention model (Hugging Face), the speaker and the judge (Ollama) do not fit on one GPU
together, so each stage loads only the model it needs.

In [ ]:
import torch, gc, time, requests, numpy as np
from PIL import Image as PILImage
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

HF_MODEL      = "Qwen/Qwen2.5-VL-7B-Instruct"
OLLAMA_URL    = "http://localhost:11434/api/generate"
SPEAKER_MODEL = "qwen3-vl:30b-a3b-instruct"     # vision speaker
JUDGE_MODEL   = "qwen3:8b"                        # text judge

def _gpu_used():
    if not torch.cuda.is_available(): return 0.0
    f,t = torch.cuda.mem_get_info(); return (t-f)/1e9

# ── HF (transformers) model ────────────────────────────────────────────────
def load_hf_model(force=False):
    """Load Qwen2.5-VL into globals model/processor (eager attn). Idempotent."""
    global model, processor
    if not force and globals().get("model") is not None and globals().get("processor") is not None:
        print(f"HF model already loaded | GPU {_gpu_used():.1f} GB"); return model, processor
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        HF_MODEL, dtype=torch.bfloat16, device_map="auto", attn_implementation="eager")
    processor = AutoProcessor.from_pretrained(HF_MODEL)
    print(f"loaded HF {HF_MODEL} | eager attn | GPU {_gpu_used():.1f} GB")
    return model, processor

def free_hf_model():
    """Drop the HF model/processor and clear VRAM. Safe if not loaded."""
    global model, processor
    for _n in ["model","processor"]:
        if globals().get(_n) is not None:
            globals()[_n] = None
            try: del globals()[_n]
            except Exception: pass
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache(); torch.cuda.synchronize()
    print(f"freed HF model | GPU {_gpu_used():.1f} GB")

# ── Ollama models (served process; load/unload via keep_alive) ──────────────
def load_ollama(model_name, wait=2):
    """Warm up an Ollama model (a tiny request makes Ollama load it). keep_alive 30m."""
    try:
        requests.post(OLLAMA_URL, json={"model":model_name,"prompt":"ok","stream":False,
                                        "keep_alive":"30m","options":{"num_predict":1}}, timeout=120)
        time.sleep(wait)
        print(f"loaded Ollama: {model_name} | GPU {_gpu_used():.1f} GB")
    except Exception as ex:
        print(f"load {model_name} failed: {str(ex)[:50]}")

def unload_ollama(model_name, wait=5):
    """Ask Ollama to drop a model now (keep_alive 0). Async, so we pause."""
    try:
        requests.post(OLLAMA_URL, json={"model":model_name,"prompt":"","keep_alive":0}, timeout=60)
        time.sleep(wait)
        print(f"unloaded Ollama: {model_name} | GPU {_gpu_used():.1f} GB")
    except Exception as ex:
        print(f"unload {model_name} skipped: {str(ex)[:50]}")

# ── free everything (clean slate) ──────────────────────────────────────────
def free_all_models():
    """Free HF + both Ollama models. Call at the start of a run for a clean GPU."""
    free_hf_model()
    for m in [SPEAKER_MODEL, JUDGE_MODEL]:
        unload_ollama(m, wait=2)
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f"── all models freed | GPU {_gpu_used():.1f} GB ──")

print("model manager ready: load_hf_model / free_hf_model / load_ollama / unload_ollama / free_all_models")

## 2. Synthetic factory-monitoring dataset

Images are rendered from three templates in `Scene_template/` (normal operation, technician, fire) with a
random room nameplate and machine status panel; see `factory_dataset/generate_factory_dataset.py`.
By default the released 500-image dataset in `factory_dataset/` is loaded (download instructions in the README).
Set `USE_RELEASED_DATASET = False` to render a fresh set instead.

In [ ]:
from collections import Counter
from generate_factory_dataset import (load_dataset, build_panel_records, answer_for_question,
                                      QUESTIONS, TECHNICIAN_Q, EVENT_OF_TEMPLATE)

USE_RELEASED_DATASET = True
N_PANELS     = 300       # images used in the paper
DATASET_SEED = None      # only used when rendering a fresh dataset

if USE_RELEASED_DATASET:
    panel_records = load_dataset(n_panels=N_PANELS)
else:
    panel_records = build_panel_records(N_PANELS, seed=DATASET_SEED)

ACTIVE_QUESTION_TYPES = list(QUESTIONS)     # the 7 tasks of Table I

print(f"{len(panel_records)} images | templates: {dict(Counter(p['meta']['template'] for p in panel_records))}")
print("tasks:")
for q in ACTIVE_QUESTION_TYPES:
    print("  -", q)

In [ ]:
# one image per event with its ground truth
picks = [next(p for p in panel_records if p["meta"]["template"] == t) for t in ["clear", "human", "fire"]]
fig, axes = plt.subplots(1, 3, figsize=(22, 9))
for ax, p in zip(axes, picks):
    ax.imshow(p["img"]); ax.axis("off")
    ax.set_title(f"{EVENT_OF_TEMPLATE[p['meta']['template']]}  (panel {p['panel_id']})", fontsize=13, fontweight="bold")
    lines = [f"{q}  →  {answer_for_question(p['meta'], q)}" for q in ACTIVE_QUESTION_TYPES]
    ax.text(0.5, -0.02, "\n".join(lines), transform=ax.transAxes,
            ha="center", va="top", fontsize=9, family="monospace")
plt.tight_layout(); plt.show()

In [ ]:
# resolve tasks by text so nothing depends on the question order
def qid_of(question):
    """Return the index of `question` in ACTIVE_QUESTION_TYPES, or raise."""
    try:
        return ACTIVE_QUESTION_TYPES.index(question)
    except ValueError:
        raise ValueError(f"Question not in ACTIVE_QUESTION_TYPES:\n  {question!r}\n"
                         f"current list:\n  " + "\n  ".join(ACTIVE_QUESTION_TYPES))

FIRE_Q = "Is there a fire?"
QID_FIRE = qid_of(FIRE_Q)
QID_TECH = qid_of(TECHNICIAN_Q)          # TECHNICIAN_Q defined in the dataset cell

## 3. SCI estimation via zero-shot learning (Algorithm 1)

A frozen **Qwen2.5-VL-7B** answers each task query with an `<answer>…</answer><where>…</where>` prompt.
The decoder attention of the `<where>` tokens onto the image tokens (mid-layer band 0.4L–0.7L, head mean,
max over tokens, attention-sink clipping, Gaussian blur, grey dilation) gives the per-task map $A_q$.
The maps are weighted by the task-criticality score $\omega_q$, max-combined, thresholded at $\tau=0.5$
and nearest-upsampled to the pixel grid, giving the transmit tensor $T=(R,G,B,S,\text{pos})$.

Eager attention is required (flash / SDPA do not return attention weights). Loading the 7B model in bfloat16
needs ~16 GB of GPU memory, so stop or unload Ollama first.

In [ ]:
load_hf_model()

In [ ]:
# Algorithm 1, lines 2-6: (image, question) → per-task map A_q on the Hm×Wm token grid
import torch, numpy as np, re
from scipy.ndimage import gaussian_filter, grey_dilation

ATTENTION_PROMPT = """Look at the image and answer using only what is visible.

Question: {question}

First give the exact answer value. Then describe WHERE in the image you see the
evidence for that answer — name the object or area and its physical location.
Do not name a room; describe the visible location in the image ie where it is in the image.

Format your response exactly as:
<answer>VALUE</answer>
<where>where you see the evidence</where>"""

def _find_span(pieces, open_tag, close_tag):
    start = end = None
    for i, p in enumerate(pieces):
        if open_tag in p and start is None: start = i + 1
        if close_tag in p and start is not None and end is None: end = i; break
    return start, end

def attention_heatmap(image, question, layer_frac=(0.4, 0.7), head_reduce="mean",
                      token_pool="max", sink_drop=4, smooth=1.2, dilate=3, verbose=False):
    prompt = ATTENTION_PROMPT.format(question=question)
    messages = [{"role":"user","content":[
        {"type":"image","image":image}, {"type":"text","text":prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)

    T, Hg, Wg = inputs["image_grid_thw"][0].tolist()
    merge = getattr(processor.image_processor, "merge_size", 2)
    Hm, Wm = Hg // merge, Wg // merge; n_vis = T * Hm * Wm
    ids = inputs["input_ids"][0].tolist()
    vs = processor.tokenizer.convert_tokens_to_ids("<|vision_start|>")
    ve = processor.tokenizer.convert_tokens_to_ids("<|vision_end|>")
    s = ids.index(vs) + 1; e = ids.index(ve); assert e - s == n_vis

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=48, do_sample=False,
                             output_attentions=True, return_dict_in_generate=True)
    gen_ids = out.sequences[0, inputs["input_ids"].shape[1]:]
    full = processor.batch_decode(gen_ids[None], skip_special_tokens=True)[0].strip()
    ans_m = re.search(r"<answer>(.*?)</answer>", full, re.S)
    whr_m = re.search(r"<where>(.*?)</where>", full, re.S)
    answer = (ans_m.group(1).strip() if ans_m else full.split("\n")[0])[:30]
    where  = (whr_m.group(1).strip() if whr_m else "")[:40]

    piece = [processor.tokenizer.decode([t]) for t in gen_ids.tolist()]
    ws, we_ = _find_span(piece, "<where>", "</where>")
    if ws is None or we_ is None or we_ <= ws:
        ws, we_ = _find_span(piece, "<answer>", "</answer>")
    if ws is None or we_ is None or we_ <= ws:
        ws, we_ = 0, len(piece)
    steps = [i for i in range(ws, we_) if i < len(out.attentions)]
    if not steps: steps = [len(out.attentions)-1]
    if verbose: print(f"  [{question[:30]}…] A={answer!r} WHERE={where!r} | {len(steps)} tok")

    maps = []
    for st in steps:
        step = out.attentions[st]; L = len(step)
        lo, hi = int(L*layer_frac[0]), max(int(L*layer_frac[1]), int(L*layer_frac[0])+1)
        sel = torch.stack(step[lo:hi]).mean(0)[0]
        a = sel.mean(0) if head_reduce=="mean" else sel.max(0).values
        maps.append(a[-1].float().cpu().numpy()[s:e])
    M = np.stack(maps, 0)
    img_attn = M.max(0) if token_pool=="max" else M.mean(0)

    if sink_drop > 0:
        thr = np.sort(img_attn)[-sink_drop-1]; img_attn = np.clip(img_attn, None, thr)
    grid = img_attn.reshape(Hm, Wm).astype(np.float32)
    raw_peak = float(grid.max() / (grid.mean() + 1e-8))
    if smooth and smooth > 0: grid = gaussian_filter(grid, sigma=smooth)
    if dilate and dilate > 1: grid = grey_dilation(grid, size=(dilate, dilate))
    heat = (grid - grid.min()) / (np.ptp(grid) + 1e-8)
    return answer, heat, raw_peak

print("attention_heatmap ready (Algorithm 1)")

Check: each task's map should land on its evidence (nameplate, status panel, flame, technician).

In [ ]:
import random
SEED = random.randrange(2**32)
rng = random.Random(SEED)
print(f"[Algorithm 1 debug] SEED = {SEED}")

def pick(t): return rng.choice([p for p in panel_records if p["meta"]["template"] == t])
img_human, img_clear, img_fire = pick("human")["img"], pick("clear")["img"], pick("fire")["img"]

# binary presence questions → test both YES/NO images (resolved by identity)
BINARY = {FIRE_Q:       (img_fire,  img_clear),
          TECHNICIAN_Q: (img_human, img_clear)}

tests = []
for q in ACTIVE_QUESTION_TYPES:
    if q in BINARY:
        pos_img, neg_img = BINARY[q]
        short = "fire" if q == FIRE_Q else "technician"
        tests.append((pos_img, q, f"{short} = POS"))
        tests.append((neg_img, q, f"{short} = NEG"))
    else:
        tests.append((img_human, q, q[:22]))

TUNE = dict(smooth=1.2, dilate=3, sink_drop=4, token_pool="max")

n=len(tests); cols=2; rows=(n+cols-1)//cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 4.4*rows)); axes=np.array(axes).reshape(-1)
results=[]
for i,(img,q,tag) in enumerate(tests):
    ans, heat, peak = attention_heatmap(img, q, **TUNE)
    npimg=np.array(img.convert("RGB")); H,W=npimg.shape[:2]
    hu=np.array(PILImage.fromarray((heat*255).astype(np.uint8)).resize((W,H),PILImage.BILINEAR))/255.0
    ax=axes[i]; ax.imshow(npimg); ax.imshow(hu,cmap="jet",alpha=0.45); ax.axis("off")
    ax.set_title(f"{tag}\nA:{ans[:28]!r} | peak={peak:.1f}", fontsize=9)
    results.append((tag,ans,peak))
for j in range(n,len(axes)): axes[j].axis("off")
plt.suptitle(f"Algorithm 1 — attention_heatmap sweep (SEED={SEED})  {TUNE}", fontsize=13, y=1.005)
plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()
print("\nRanked by concentration:")
for tag,a,p in sorted(results, key=lambda r:-r[2]):
    print(f"  peak={p:5.1f} | {tag:22} | A:{a[:30]!r}")

In [ ]:
# Algorithm 1, line 8: weight each A_q by its score and max-combine
import numpy as np

# qids index into ACTIVE_QUESTION_TYPES. Critical questions → score 2.
CRITICAL = {i: 2 for i in range(len(ACTIVE_QUESTION_TYPES))}

def importance_map(image, critical=CRITICAL, combine="max", **tune):
    """Run each critical question, weight by score, combine → one token-grid map.
       combine='max' (union) or 'sum' (shared-importance)."""
    per_q = {}
    stacked = []
    for qid, score in critical.items():
        q = ACTIVE_QUESTION_TYPES[qid]
        ans, heat, peak = attention_heatmap(image, q, **tune)   # heat 0..1, Hm×Wm
        per_q[qid] = (q, ans, heat, peak)
        stacked.append(score * heat)                            # weight by score
    S = np.stack(stacked, 0)
    combined = S.max(0) if combine == "max" else S.sum(0)
    combined = (combined - combined.min()) / (np.ptp(combined) + 1e-8)
    return combined, per_q

print("importance_map ready")

In [ ]:
import random
SEED = random.randrange(2**32)
rng = random.Random(SEED)
print(f"[importance_map debug] SEED = {SEED}")

# which questions are critical for this debug (qid: score). Default = all @ 2.
CRITICAL_DEBUG = {i: 2 for i in range(len(ACTIVE_QUESTION_TYPES))}
TUNE = dict(smooth=1.2, dilate=3, sink_drop=4, token_pool="max")

print("critical questions:")
for qid, sc in CRITICAL_DEBUG.items():
    print(f"  q{qid} (score {sc}): {ACTIVE_QUESTION_TYPES[qid]}")

def show_importance(image, title=""):
    combined, per_q = importance_map(image, CRITICAL_DEBUG, combine="max", **TUNE)
    npimg = np.array(image.convert("RGB")); H, W = npimg.shape[:2]
    n = len(per_q) + 1; cols = 3; rows = (n + cols - 1)//cols
    fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 4*rows)); axes = np.array(axes).reshape(-1)
    for i, (qid, (q, ans, heat, peak)) in enumerate(per_q.items()):
        hu = np.array(PILImage.fromarray((heat*255).astype(np.uint8)).resize((W,H),PILImage.BILINEAR))/255.0
        ax = axes[i]; ax.imshow(npimg); ax.imshow(hu, cmap="jet", alpha=0.45); ax.axis("off")
        ax.set_title(f"q{qid}: {q[:20]}\nA:{ans[:16]!r} pk={peak:.0f}", fontsize=8)
    ax = axes[len(per_q)]
    hu = np.array(PILImage.fromarray((combined*255).astype(np.uint8)).resize((W,H),PILImage.BILINEAR))/255.0
    ax.imshow(npimg); ax.imshow(hu, cmap="jet", alpha=0.55); ax.axis("off")
    ax.set_title(f"★ COMBINED (max)\nqids: {list(CRITICAL_DEBUG.keys())}", fontsize=9, fontweight="bold")
    for j in range(len(per_q)+1, len(axes)): axes[j].axis("off")
    plt.suptitle(f"importance_map — {title}  (SEED={SEED})", fontsize=13, y=1.005)
    plt.tight_layout(rect=[0,0,1,0.99]); plt.show()

for template in ["human", "fire"]:
    img = rng.choice([p for p in panel_records if p["meta"]["template"]==template])["img"]
    show_importance(img, title=f"{template} panel")

In [ ]:
# Algorithm 1, line 9: token-grid map → discrete SCI levels (binary: 2 = critical, 1 = normal)
import numpy as np

TAU = 0.5

def attention_to_sci(att, tau=TAU, levels=None):
    if levels is None:
        return np.where(att > tau, 2, 1).astype(int)
    return (np.digitize(att, levels) + 1).astype(int)

print("attention_to_sci ready")

In [ ]:
# Algorithm 1 (full): image → transmit tensor T = (R, G, B, S, pos)
import numpy as np
from PIL import Image as PILImage

# critical question set {qid: score}; qids index ACTIVE_QUESTION_TYPES.
# Critical questions → score 2 (normal questions are simply not included).
CRITICAL = {i: 2 for i in range(len(ACTIVE_QUESTION_TYPES))}

def BuildSCI(image, critical=CRITICAL, tau=TAU, levels=None, combine="max", **tune):
    """
    image    : PIL RGB image (full resolution)
    critical : {qid: score} critical questions (qids index ACTIVE_QUESTION_TYPES)
    tau      : binary threshold (used when levels is None)
    levels   : ascending vector for graded SCI (else binary)
    Returns
      T       : int64 (H, W, 5) = (R, G, B, S, pos)
      A_up    : (H, W) continuous importance map, NEAREST-upsampled   (debug)
      per_q   : {qid: (question, answer, heat, peak)}                 (debug)
      sci_tok : (Hm, Wm) token-grid SCI levels before upsample        (debug)
    """
    npimg = np.array(image.convert("RGB")); H, W = npimg.shape[:2]

    # 1-2. combine (token grid) then threshold (token grid)
    A, per_q = importance_map(image, critical, combine=combine, **tune)    # Hm×Wm, [0,1]
    sci_tok  = attention_to_sci(A, tau=tau, levels=levels)                 # Hm×Wm, {1,2,...}

    # 3. NEAREST-upsample the DISCRETE mask to pixels  (tight, block-aligned)
    S = np.array(PILImage.fromarray(sci_tok.astype(np.uint8))
                 .resize((W, H), PILImage.NEAREST)).astype(np.int64)

    # 4. pack per-pixel tensor T = (R, G, B, S, pos), pos = row*W + col
    rows, cols = np.meshgrid(np.arange(H), np.arange(W), indexing="ij")
    pos = (rows * W + cols).astype(np.int64)
    T = np.zeros((H, W, 5), dtype=np.int64)
    T[..., 0:3] = npimg
    T[..., 3]   = S
    T[..., 4]   = pos

    # extra: continuous map upsampled the same way (for debug overlay)
    A_up = np.array(PILImage.fromarray((A*255).astype(np.uint8))
                    .resize((W, H), PILImage.NEAREST)) / 255.0
    return T, A_up, per_q, sci_tok

print("BuildSCI ready (returns T = (R,G,B,S,pos))")

Check the tensor contract of $T$ and show the SCI channel for each event.

In [ ]:
import random
from matplotlib.colors import ListedColormap, BoundaryNorm
SEED = random.randrange(2**32)
rng = random.Random(SEED)
print(f"[BuildSCI debug] SEED = {SEED}")

CRITICAL_DEBUG = {i: 2 for i in range(len(ACTIVE_QUESTION_TYPES))}
TUNE = dict(smooth=1.2, dilate=3, sink_drop=4, token_pool="max")

TEMPLATES = ["human", "fire", "clear"]
sci_cmap = ListedColormap(["#3b4cc0", "#d62728"]); norm = BoundaryNorm([0.5,1.5,2.5], 2)

fig, axes = plt.subplots(len(TEMPLATES), 4, figsize=(22, 5.2*len(TEMPLATES)))
axes = np.atleast_2d(axes)

for r, template in enumerate(TEMPLATES):
    panel = rng.choice([p for p in panel_records if p["meta"]["template"]==template])
    image = panel["img"]
    T, A_up, per_q, sci_tok = BuildSCI(image, CRITICAL_DEBUG, tau=TAU, **TUNE)

    npimg = np.array(image.convert("RGB")); H, W = npimg.shape[:2]

    # ── HARD CONTRACT CHECKS (fail loudly) ──
    assert T.shape == (H, W, 5),                 f"[{template}] T shape {T.shape} != {(H,W,5)}"
    assert T.dtype == np.int64,                  f"[{template}] T dtype {T.dtype} != int64"
    assert np.array_equal(T[...,0:3], npimg),    f"[{template}] T RGB != source image"
    S = T[...,3]; pos = T[...,4]
    assert set(np.unique(S)).issubset({1,2}),    f"[{template}] S levels {np.unique(S)} not ⊆ {{1,2}}"
    expected_pos = np.arange(H)[:,None]*W + np.arange(W)[None,:]
    assert np.array_equal(pos, expected_pos),    f"[{template}] pos != raster row*W+col"
    n_crit = int((S==2).sum()); n_norm = int((S==1).sum())
    assert n_crit + n_norm == H*W,               f"[{template}] |C|+|M| != N"
    # pos ↔ RGB self-consistency for a critical pixel (what Encode/Decode rely on)
    if n_crit > 0:
        cp = np.argwhere(S==2)[0]; p = int(pos[cp[0],cp[1]])
        assert p == cp[0]*W + cp[1],             f"[{template}] pos mismatch at critical pixel"
        assert (p // W, p % W) == (cp[0], cp[1]), f"[{template}] pos→(row,col) inverse broken"

    crit_pct = 100*n_crit/(H*W)
    print(f"✓ [{template:5}] T{T.shape} int64 | RGB✓ | S∈{{1,2}}✓ | pos raster✓ | "
          f"|C|={n_crit} ({crit_pct:.1f}%)  |M|={n_norm}")

    # ── VISUAL: image | SCI overlay | S channel | pos channel ──
    axes[r,0].imshow(npimg); axes[r,0].set_title(f"{template} — original ({W}×{H})"); axes[r,0].axis("off")
    axes[r,1].imshow(npimg); im=axes[r,1].imshow(S, cmap=sci_cmap, norm=norm, alpha=0.5)
    axes[r,1].set_title(f"SCI overlay — {crit_pct:.1f}% critical"); axes[r,1].axis("off")
    cbar=fig.colorbar(im, ax=axes[r,1], ticks=[1,2], fraction=0.046, pad=0.04)
    cbar.ax.set_yticklabels(["1 normal","2 critical"])
    axes[r,2].imshow(S, cmap=sci_cmap, norm=norm)
    axes[r,2].set_title("T[...,3] = S channel"); axes[r,2].axis("off")
    im2=axes[r,3].imshow(pos, cmap="viridis")
    axes[r,3].set_title("T[...,4] = pos (row*W+col)"); axes[r,3].axis("off")
    fig.colorbar(im2, ax=axes[r,3], fraction=0.046, pad=0.04)

plt.suptitle(f"BuildSCI — tensor T = (R,G,B,S,pos) verification  (SEED={SEED})", fontsize=13, y=1.005)
plt.tight_layout(rect=[0,0,1,0.99]); plt.show()
print("\n✓ All tensor contracts passed — T is valid input for the transmitter (Sec. III-B).")

Choose any set of critical tasks `{qid: score}` and look at the resulting SCI channel.

In [ ]:
import random
from matplotlib.colors import ListedColormap, BoundaryNorm
SEED = random.randrange(2**32)
rng = random.Random(SEED)
print(f"[BuildSCI pipeline debug] SEED = {SEED}")

# critical tasks (qid: score). Examples:
#    {QID_FIRE: 2, QID_TECH: 2}                        → the two hazards
#    {qid_of("Which room is this?"): 2, QID_FIRE: 2}   → room + fire
#    {i: 2 for i in range(len(ACTIVE_QUESTION_TYPES))} → all tasks
CRITICAL_SET = {qid_of("Which room is this?"): 2}

TUNE = dict(smooth=1.2, dilate=3, sink_drop=4, token_pool="max")

# ── print the critical questions in full (text + score) ──
print(f"\nCritical questions — {len(CRITICAL_SET)} total:")
for qid, score in CRITICAL_SET.items():
    print(f"  q{qid}  (score {score}):  {ACTIVE_QUESTION_TYPES[qid]}")
print()

TEMPLATES = ["human", "fire", "clear"]
imgs = [(t, rng.choice([p for p in panel_records if p["meta"]["template"]==t])["img"])
        for t in TEMPLATES]

sci_cmap = ListedColormap(["#3b4cc0", "#d62728"])
norm = BoundaryNorm([0.5,1.5,2.5], sci_cmap.N)

rows = len(imgs)
fig, axes = plt.subplots(rows, 3, figsize=(19, 5.2*rows)); axes = np.atleast_2d(axes)

for r, (template, image) in enumerate(imgs):
    # single BuildSCI call gives everything: tensor T + debug maps
    T, A_up, per_q, sci_tok = BuildSCI(image, CRITICAL_SET, tau=TAU, **TUNE)
    npimg = np.array(image.convert("RGB")); H, W = npimg.shape[:2]
    S = T[..., 3]
    crit = (S == 2).mean()

    axes[r,0].imshow(npimg); axes[r,0].set_title(f"{template} — image"); axes[r,0].axis("off")
    axes[r,1].imshow(npimg); axes[r,1].imshow(A_up, cmap="jet", alpha=0.5)
    axes[r,1].set_title("ATTENTION channel (continuous 0-1)"); axes[r,1].axis("off")
    axes[r,2].imshow(npimg); im=axes[r,2].imshow(S, cmap=sci_cmap, norm=norm, alpha=0.5)
    axes[r,2].set_title(f"SCI channel (τ={TAU}) — {crit:.0%} critical"); axes[r,2].axis("off")
    cbar = fig.colorbar(im, ax=axes[r,2], ticks=[1,2], fraction=0.046, pad=0.04)
    cbar.ax.set_yticklabels(["1 normal","2 critical"])

q_lines = "  |  ".join(f"q{qid}(s{score}): {ACTIVE_QUESTION_TYPES[qid][:22]}"
                       for qid, score in CRITICAL_SET.items())
plt.suptitle(f"BuildSCI pipeline — critical set: {q_lines}  (SEED={SEED})", fontsize=12, y=1.005)
plt.tight_layout(rect=[0,0,1,0.99]); plt.show()

## 4. Tiered transmission over 5G NR OFDM (Sec. III-B)

Sionna link: 30 kHz SCS, 14 OFDM symbols, 128 subcarriers, 3.5 GHz, 3GPP TDL-A (100 ns), perfect CSI,
LMMSE equalization, fixed $E_s/N_0$ for every modulation order.

* **Base layer**: whole image, bilinear downsampling by $D$, uncoded $2^{m_b}$-QAM, upsampled at the receiver.
* **Protection tier**: one self-addressed unit `[pos | R | G | B]` per task-causal pixel, terminated rate-1/3
  convolutional code ($\kappa=7$), BPSK, soft-input Viterbi. Decoded pixels overwrite the base layer.

In [ ]:
import numpy as np, torch, gc
from PIL import Image as PILImage
from sionna.phy.mapping import Mapper, Demapper
from sionna.phy.channel import GenerateOFDMChannel, ApplyOFDMChannel
from sionna.phy.channel.tr38901 import TDL
from sionna.phy.ofdm import ResourceGrid, ResourceGridMapper, LMMSEEqualizer
from sionna.phy.mimo import StreamManagement
from sionna.phy.utils import hard_decisions
from sionna.phy.fec.conv import ConvEncoder, ViterbiDecoder

_dev = "cuda" if torch.cuda.is_available() else "cpu"

# ── OFDM / channel config ──
FFT_SIZE=128; NUM_OFDM_SYM=14; SUBCARRIER_SPACING=30e3
NUM_TX=1; NUM_STREAMS=1
TDL_MODEL="A"; DELAY_SPREAD=100e-9; CARRIER_FREQ=3.5e9; MIN_SPEED=0.0; MAX_SPEED=0.0

RG=ResourceGrid(num_ofdm_symbols=NUM_OFDM_SYM,fft_size=FFT_SIZE,
                subcarrier_spacing=SUBCARRIER_SPACING,num_tx=NUM_TX,
                num_streams_per_tx=NUM_STREAMS,cyclic_prefix_length=6,
                pilot_pattern=None,num_guard_carriers=(0,0),dc_null=False)
DATA_RE_PER_GRID=RG.num_data_symbols
SM=StreamManagement(np.array([[1]]),1)

_tdl=TDL(model=TDL_MODEL,delay_spread=DELAY_SPREAD,carrier_frequency=CARRIER_FREQ,
         min_speed=MIN_SPEED,max_speed=MAX_SPEED)
gen_chan=GenerateOFDMChannel(_tdl,RG,normalize_channel=True)
apply_chan=ApplyOFDMChannel()
rg_mapper=ResourceGridMapper(RG)
lmmse_equ=LMMSEEqualizer(RG,SM)

# ── mapper cache: m=1 → BPSK (pam), even m → QAM ──
_MC={}
def mapper_for(m):
    if m not in _MC:
        if m==1:
            _MC[m]=(Mapper(constellation_type='pam',num_bits_per_symbol=1),
                    Demapper('app',constellation_type='pam',num_bits_per_symbol=1))
        else:
            _MC[m]=(Mapper('qam',m),Demapper('app','qam',m))
    return _MC[m]

# ── conv code for criticals (rate 1/3, BPSK) ──
CONV_RATE=1/3; CONV_CL=7
conv_enc=ConvEncoder(rate=CONV_RATE,constraint_length=CONV_CL,terminate=True)
conv_dec=ViterbiDecoder(encoder=conv_enc,method="soft_llr")

# ── config knobs ──
M_BASE=8; M_CRIT=1; DS=4          # 256-QAM base, BPSK criticals, ↓4× base

def _esn0_no(esn0_db):
    return torch.tensor(10**(-esn0_db/10.0),dtype=torch.float32,device=_dev)
def _bits_msb(v,w):
    v=np.asarray(v,np.int64); s=np.arange(w-1,-1,-1,np.int64)
    return ((v[:,None]>>s)&1).astype(np.uint8)
def _bits_to_int_rows(b):
    w=b.shape[1]; return (b.astype(np.int64)*(1<<np.arange(w-1,-1,-1))).sum(1)

def send_bits_esn0(bits_flat, m, esn0_db):
    """Uncoded bits at QAM order m over genie channel, fixed Es/N0. Returns recovered bits."""
    mp,dm=mapper_for(m); C=DATA_RE_PER_GRID; bpg=C*m
    ng=int(np.ceil(bits_flat.size/bpg)); pad=ng*bpg-bits_flat.size
    b=torch.tensor(np.concatenate([bits_flat,np.zeros(pad,np.uint8)]),
                   dtype=torch.float32,device=_dev).reshape(ng,NUM_TX,NUM_STREAMS,bpg)
    no=_esn0_no(esn0_db); out=[]
    for s in range(0,ng,512):
        x=mp(b[s:s+512]); xr=rg_mapper(x)
        h=gen_chan(batch_size=x.shape[0]); y=apply_chan(xr,h,no)
        xh,ne=lmmse_equ(y,h,torch.zeros_like(no),no)
        out.append(hard_decisions(dm(xh,ne)).to(torch.uint8).reshape(-1).cpu())
    return torch.cat(out).numpy()[:bits_flat.size]

print("transmission foundations ready")
print(f"  base ↓{DS}× @ {2**M_BASE}QAM | criticals conv r={CONV_RATE:.2f} @ {'BPSK' if M_CRIT==1 else str(2**M_CRIT)+'QAM'}")
print(f"  grid C={DATA_RE_PER_GRID} | Es/N0 metric (fixed no)")

In [ ]:
def _send_base_x(npimg, esn0_db, m_base, ds):
    """Base layer: downsample by ds, uncoded 2^m_base-QAM, upsample back to full resolution."""
    H,W=npimg.shape[:2]; Hd,Wd=H//ds,W//ds
    small=np.array(PILImage.fromarray(npimg).resize((Wd,Hd),PILImage.BILINEAR),np.uint8)
    bits=np.unpackbits(small.reshape(-1))
    bhat=send_bits_esn0(bits,m_base,esn0_db)
    rec_small=np.packbits(bhat)[:small.size].reshape(Hd,Wd,3)
    return np.array(PILImage.fromarray(rec_small).resize((W,H),PILImage.BILINEAR),np.uint8)

def _send_crit_conv(crit_idx, crit_rgb, POS, esn0_db, chunk=20000):
    """Critical layer: [POS|R|G|B] per critical pixel → conv rate-1/3 → BPSK → channel → Viterbi."""
    mp,dm=mapper_for(M_CRIT); Lc=len(crit_idx); Uunit=POS+24; no=_esn0_no(esn0_db)
    conv_dec_local=ViterbiDecoder(encoder=conv_enc, method="soft_llr")   # fresh per call → correct shape
    pos_a=np.empty(Lc,np.int64);R_a=np.empty(Lc,np.int64);G_a=np.empty(Lc,np.int64);B_a=np.empty(Lc,np.int64)
    for s in range(0,Lc,chunk):
        e=min(s+chunk,Lc); idx=crit_idx[s:e]; rgb=crit_rgb[s:e]
        u=np.concatenate([_bits_msb(idx,POS),_bits_msb(rgb[:,0],8),
                          _bits_msb(rgb[:,1],8),_bits_msb(rgb[:,2],8)],1)
        u=torch.tensor(u,dtype=torch.float32,device=_dev)
        c=conv_enc(u); n_coded=c.shape[1]; cflat=c.reshape(-1)
        n_units=e-s; total_coded=n_units*n_coded
        C=DATA_RE_PER_GRID; bpg=C*M_CRIT; ng=int(np.ceil(cflat.numel()/bpg)); pad=ng*bpg-cflat.numel()
        cb=torch.cat([cflat,torch.zeros(pad,device=_dev)]).reshape(ng,NUM_TX,NUM_STREAMS,bpg)
        llr=[]
        for g in range(0,ng,512):
            x=mp(cb[g:g+512]); xr=rg_mapper(x)
            h=gen_chan(batch_size=x.shape[0]); y=apply_chan(xr,h,no)
            xh,ne=lmmse_equ(y,h,torch.zeros_like(no),no); llr.append(dm(xh,ne).reshape(-1))
        llr=torch.cat(llr)[:total_coded].reshape(n_units,n_coded)
        uhat=conv_dec_local(llr).to(torch.uint8).cpu().numpy()[:, :Uunit]
        pos_a[s:e]=_bits_to_int_rows(uhat[:,:POS]);R_a[s:e]=_bits_to_int_rows(uhat[:,POS:POS+8])
        G_a[s:e]=_bits_to_int_rows(uhat[:,POS+8:POS+16]);B_a[s:e]=_bits_to_int_rows(uhat[:,POS+16:POS+24])
        del u,c,cb,llr,uhat; gc.collect(); torch.cuda.empty_cache()
    return pos_a,R_a,G_a,B_a

def _sci_mask_allq(img):
    """All 7 tasks critical → (T, crit_pix, crit_idx, crit_rgb, POS)."""
    CRITICAL_SET={i:2 for i in range(len(ACTIVE_QUESTION_TYPES))}
    T,_,_,_=BuildSCI(img, CRITICAL_SET, tau=TAU, smooth=1.2,dilate=3,sink_drop=4,token_pool="max")
    H,W,_=T.shape; N=H*W
    crit_pix=(T[...,3]==2)
    crit_idx=np.where(crit_pix.reshape(-1))[0]
    crit_rgb=np.array(img,np.uint8).reshape(-1,3)[crit_idx]
    POS=int(np.ceil(np.log2(N)))               # B_a = ceil(log2 P) address bits
    return T,crit_pix,crit_idx,crit_rgb,POS

def _to_uint8(img):
    return np.array(img.convert("RGB"),np.uint8) if hasattr(img,"convert") else np.asarray(img,np.uint8)

def transmit_baseline_x(img, esn0_db, m_base, ds):
    """Conventional scheme: whole image downsampled by ds, uncoded 2^m_base-QAM."""
    return _send_base_x(_to_uint8(img), esn0_db, m_base, ds)

def transmit_sci_x(img, esn0_db, m_base, ds, sci_cache):
    """STAR: base layer + conv-coded BPSK critical layer overwriting the critical pixels.
       sci_cache = (crit_idx, crit_rgb, POS, crit_pix) from _sci_mask_allq."""
    npimg=_to_uint8(img); H,W=npimg.shape[:2]; N=H*W
    crit_idx,crit_rgb,POS,crit_pix=sci_cache
    base_img=_send_base_x(npimg, esn0_db, m_base, ds)
    pos,R,G,B=_send_crit_conv(crit_idx,crit_rgb,POS,esn0_db)
    v=pos<N
    sci=base_img.reshape(-1,3).copy(); sci[pos[v]]=np.stack([R,G,B],1)[v]
    return sci.reshape(H,W,3), crit_pix

# shorthands with the default configuration (M_BASE, DS)
def transmit_baseline(img, esn0_db):
    return transmit_baseline_x(img, esn0_db, M_BASE, DS)

def transmit_sci(img, esn0_db, sci_cache=None):
    if sci_cache is None:
        _,crit_pix,crit_idx,crit_rgb,POS=_sci_mask_allq(PILImage.fromarray(_to_uint8(img)))
        sci_cache=(crit_idx,crit_rgb,POS,crit_pix)
    return transmit_sci_x(img, esn0_db, M_BASE, DS, sci_cache)

print("transmit_baseline_x / transmit_sci_x ready")

Sanity check on one fire image: at 40 dB the task-causal pixels must be recovered exactly, and as $E_s/N_0$
drops STAR should keep the critical regions clean while the downsampled baseline degrades.

In [ ]:
import random, matplotlib.pyplot as plt
SEED=random.randrange(2**32); rng=random.Random(SEED)
print(f"[verify] SEED={SEED}")

panel=rng.choice([p for p in panel_records if p["meta"]["template"]=="fire"])
img=panel["img"].convert("RGB"); npimg=np.array(img,np.uint8); H,W=npimg.shape[:2]; N=H*W

def psnr(a,b,mask=None):
    a=a.astype(np.float64);b=b.astype(np.float64)
    if mask is not None:a=a[mask];b=b[mask]
    e=np.mean((a-b)**2);return 99 if e==0 else 10*np.log10(255**2/e)

# build SCI mask ONCE, reuse across Es/N0 (this is the expensive VLM step)
_,crit_pix,crit_idx,crit_rgb,POS=_sci_mask_allq(img)
cache=(crit_idx,crit_rgb,POS,crit_pix)
print(f"{W}×{H} | ALL-Q critical={100*crit_pix.mean():.1f}% | |C|={len(crit_idx):,} | POS={POS}")

# airtime
Hd,Wd=H//DS,W//DS
base_syms=int(np.ceil(Hd*Wd*3*8/M_BASE))
n_coded=(POS+24)/CONV_RATE+(CONV_CL-1)/CONV_RATE
crit_syms=int(len(crit_idx)*n_coded/M_CRIT)
print(f"airtime: base(↓{DS}×)={base_syms:,} sym | crit(conv-BPSK)={crit_syms:,} sym | "
      f"total={base_syms+crit_syms:,} ({(base_syms+crit_syms)/base_syms:.1f}× base-only)\n")

# ── (1) noiseless sanity at 40 dB ──
b40=transmit_baseline(img,40)
s40,_=transmit_sci(img,40,sci_cache=cache)
# criticals should be exact at 40 dB (conv-coded, near-zero errors)
crit_exact=np.array_equal(s40.reshape(-1,3)[crit_idx], crit_rgb)
print(f"(1) noiseless@40dB: base blurry-PSNR={psnr(npimg,b40):.1f} | "
      f"SCI critical pixels exact={crit_exact} | SCI crit-PSNR={psnr(npimg,s40,crit_pix):.1f}")

# ── SCI-channel overlay ──
from matplotlib.colors import ListedColormap,BoundaryNorm
cm=ListedColormap(["#3b4cc0","#d62728"]); nm=BoundaryNorm([0.5,1.5,2.5],2)
fig,ax=plt.subplots(1,2,figsize=(14,5))
ax[0].imshow(npimg);ax[0].set_title(f"original {W}×{H}");ax[0].axis("off")
ax[1].imshow(npimg);im=ax[1].imshow(crit_pix.astype(int)+1,cmap=cm,norm=nm,alpha=0.45)
ax[1].set_title(f"SCI channel (ALL-Q) — {100*crit_pix.mean():.1f}% critical");ax[1].axis("off")
cb=fig.colorbar(im,ax=ax[1],ticks=[1,2],fraction=0.046);cb.ax.set_yticklabels(["normal","critical"])
plt.tight_layout();plt.show()

# ── (2) visual sweep: baseline vs SCI across Es/N0 ──
SHOW=[-8,-4,0,4,8,12,16,20]
rows=[]
for esn0 in SHOW:
    b=transmit_baseline(img,esn0)
    s,_=transmit_sci(img,esn0,sci_cache=cache)
    rows.append(dict(esn0=esn0,base=b,sci=s,
                     base_c=psnr(npimg,b,crit_pix),sci_c=psnr(npimg,s,crit_pix)))
    print(f"  Es/N0={esn0:3d} | SCI crit={rows[-1]['sci_c']:.1f} | BASE crit={rows[-1]['base_c']:.1f}")

fig,axes=plt.subplots(3,len(rows),figsize=(5.5*len(rows),12)); axes=np.atleast_2d(axes)
for j,r in enumerate(rows):
    axes[0,j].imshow(npimg);axes[0,j].axis("off");axes[0,j].set_title("original")
    axes[1,j].imshow(r["sci"]);axes[1,j].axis("off");axes[1,j].set_title(f"SCI {r['esn0']}dB\ncrit={r['sci_c']:.1f} dB")
    axes[2,j].imshow(r["base"]);axes[2,j].axis("off");axes[2,j].set_title(f"BASE {r['esn0']}dB\ncrit={r['base_c']:.1f} dB")
plt.suptitle(f"Verify: SCI (all-Q, conv-BPSK crit) vs downsampled baseline (SEED={SEED})",y=1.005,fontsize=13)
plt.tight_layout(rect=[0,0,1,0.99]); plt.show()

## 5. Semantic evaluation: speaker VLM + LLM judge

The **speaker** (Qwen3-VL-30B-A3B, served by Ollama) answers each query on the received image, and the
**judge** (Qwen3-8B, Ollama) decides whether the answer is semantically equivalent to the ground truth
(S-Acc, Sec. IV-A). S-PSNR (Eq. 6) is the PSNR over the ground-truth task-causal region only.

```bash
ollama pull qwen3-vl:30b-a3b-instruct
ollama pull qwen3:8b
```

In [ ]:
import requests, re, io, base64
import numpy as np
from PIL import Image as PILImage

OLLAMA_NUM_CTX = 8192   # OLLAMA_URL, SPEAKER_MODEL, JUDGE_MODEL come from the model-manager cell

def _img_to_b64(img):
    """PIL or uint8 array → base64 PNG (matches encode_image convention)."""
    if not hasattr(img,"save"): img=PILImage.fromarray(np.asarray(img,np.uint8))
    buf=io.BytesIO(); img.save(buf,format="PNG"); return base64.b64encode(buf.getvalue()).decode("utf-8")

def ask_vision(prompt, img, model=None, timeout=300, max_tokens=256):
    """Single-image VLM query via Ollama."""
    payload={"model":model or SPEAKER_MODEL,"prompt":prompt,"images":[_img_to_b64(img)],
             "stream":False,"keep_alive":"30m","think":False,
             "options":{"temperature":0,"top_p":1,"top_k":1,"num_ctx":OLLAMA_NUM_CTX,"num_predict":max_tokens}}
    r=requests.post(OLLAMA_URL,json=payload,timeout=timeout); d=r.json()
    if "error" in d: raise RuntimeError(d["error"])
    return d.get("response","").strip()

def ask_text(prompt, model=None, timeout=120, max_tokens=64):
    """Text-only query via Ollama (judge)."""
    payload={"model":model or JUDGE_MODEL,"prompt":prompt,"stream":False,"keep_alive":"30m","think":False,
             "options":{"temperature":0,"top_p":1,"top_k":1,"num_ctx":OLLAMA_NUM_CTX,"num_predict":max_tokens}}
    r=requests.post(OLLAMA_URL,json=payload,timeout=timeout); d=r.json()
    if "error" in d: raise RuntimeError(d["error"])
    return d.get("response","").strip()

def parse_answer_text(reply):
    """Extract <answer>...</answer>, else fall back to the raw reply."""
    m=re.search(r"<answer>(.*?)</answer>", reply, re.DOTALL|re.IGNORECASE)
    return (m.group(1) if m else reply).strip()

def truth_to_text(truth):
    if isinstance(truth,bool): return "yes" if truth else "no"
    return str(truth)

def full_image_question_answer(img, question):
    """Ask the VLM `question` about `img`. Returns (parsed_answer, raw_reply)."""
    prompt=f"""Answer the question using only the image.

QUESTION:
{question}

Rules:
- Do not explain.
- Return only the final answer.

Return only in this format:
<answer>final answer here</answer>
"""
    reply=ask_vision(prompt, img)
    return parse_answer_text(reply), reply

def llm_judge_equivalent(question, truth, model_answer):
    """Judge semantic equivalence of model_answer vs ground-truth. Returns (bool, verdict)."""
    truth_text=truth_to_text(truth)
    prompt=f"""You are grading a semantic communication result.

Decide whether the MODEL ANSWER means the same as the CORRECT ANSWER for the given question.

QUESTION:
{question}

CORRECT ANSWER:
{truth_text}

MODEL ANSWER:
{model_answer}

Rules:
- Judge only semantic equivalence.
- Ignore XML tags, capitalization, and extra words if meaning is preserved.
- Numeric percentages with/without % are equivalent (91, 91%, 91 percent).
- yes/on/true mean yes; no/off/false mean no.
- Device ID / room must match the actual value.
- Category answers must match the category.

Reply with only one word: YES or NO
"""
    verdict=ask_text(prompt, model=JUDGE_MODEL)
    return verdict.strip().upper().startswith("Y"), verdict

# ── quick connectivity + sanity test on ONE clean image ──
_p=panel_records[0]
_ans,_raw=full_image_question_answer(_p["img"].convert("RGB"), ACTIVE_QUESTION_TYPES[0])
_gt=answer_for_question(_p["meta"], ACTIVE_QUESTION_TYPES[0])
_ok,_v=llm_judge_equivalent(ACTIVE_QUESTION_TYPES[0], _gt, _ans)
print("Ollama eval infra ready.")
print(f"  Q: {ACTIVE_QUESTION_TYPES[0]}")
print(f"  VLM answer: {_ans!r} | ground truth: {_gt!r} | judge: {_v} → correct={_ok}")

## 6. STAR vs conventional QAM (Fig. 4 and Fig. 5)

60 test images (20 per template), all 7 tasks critical, base layers from BPSK to 1024-QAM.
STAR always uses $D=4$; the conventional scheme uses `CONV_DS`:

* **Fig. 4**: `CONV_DS = 4` (compressed)
* **Fig. 5**: `CONV_DS = 1` (raw image)

Stages: SCI masks (HF) → transmission + speaker (Ollama) → judge (Ollama) → aggregation. Each modulation
order is checkpointed, so an interrupted run resumes with `RESUME = True`.

In [ ]:
import random, time, pickle, os, gc, requests, numpy as np, torch, matplotlib.pyplot as plt
from PIL import Image as PILImage

# ── knobs ──
STAR_DS = 4          # STAR base downsampling (change to run STAR at any DS)
CONV_DS = 1          # conventional baseline downsampling (Fig. 4: 4, Fig. 5: 1)
M_CRIT_RUN = 1
ESN0_SWEEP = [-8,-6,-4,-2,0,2,4,8,12,16,18]
SEED_RUN = 20240612
BASE_QAMS = [(10,"1024qam"),(8,"256qam"),(6,"64qam"),(4,"16qam"),(2,"qpsk"),(1,"bpsk")]
N_PER_TEMPLATE = 20
RESUME = True

SAVE_DIR = os.path.join(RESULTS_DIR, f"STAR_DS{STAR_DS}_conv_DS{CONV_DS}")
os.makedirs(SAVE_DIR, exist_ok=True); print("SAVE_DIR:", SAVE_DIR)

def _modname(mb): return "BPSK" if mb==1 else ("QPSK" if mb==2 else f"{2**mb}-QAM")
def _gpu(): gc.collect(); (torch.cuda.empty_cache() if torch.cuda.is_available() else None)
def _mem():
    if not torch.cuda.is_available(): return ""
    f,t=torch.cuda.mem_get_info(); return f"{(t-f)/1e9:.1f}GB"
def _ckpt_path(tag): return os.path.join(SAVE_DIR,f"_ckpt_{tag}.pkl")
def psnr(a,b,mask=None):
    a=a.astype(np.float64);b=b.astype(np.float64)
    if mask is not None:a=a[mask];b=b[mask]
    e=np.mean((a-b)**2);return 99.0 if e==0 else 10*np.log10(255**2/e)

# ══ STAGE -1: clean slate ══
free_all_models()
print(f"speaker = {SPEAKER_MODEL}\n")

# ── pick balanced images ──
rng=random.Random(SEED_RUN); test_imgs=[]
for tmpl in ["fire","human","clear"]:
    cands=[p for p in panel_records if p["meta"]["template"]==tmpl]
    test_imgs+=rng.sample(cands,min(N_PER_TEMPLATE,len(cands)))
rng.shuffle(test_imgs)
N_IMAGES=len(test_imgs); QLIST=list(ACTIVE_QUESTION_TYPES)
print(f"{N_IMAGES} imgs | {len(BASE_QAMS)} QAMs × {len(ESN0_SWEEP)} SNR × {len(QLIST)} Q | STAR DS={STAR_DS}, conv DS={CONV_DS}\n")

# ══ STAGE 0: SCI masks once per image (HF Qwen2.5-VL) ══
print("[Stage 0] SCI masks...")
load_hf_model()
masks=[]; truths=[]
for ii,panel in enumerate(test_imgs):
    img=panel["img"].convert("RGB"); npimg=np.array(img,np.uint8)
    _,crit_pix,crit_idx,crit_rgb,POS=_sci_mask_allq(img)
    masks.append(dict(npimg=npimg,crit_pix=crit_pix,cache=(crit_idx,crit_rgb,POS,crit_pix),tmpl=panel["meta"]["template"]))
    truths.append({q:answer_for_question(panel["meta"],q) for q in QLIST})
    print(f"  mask {ii+1}/{N_IMAGES} ({panel['meta']['template']})", flush=True); _gpu()
free_hf_model()
print("[Stage 0] done\n")

# ══ STAGES 1+2: transmit STAR (STAR_DS) + conventional (CONV_DS) + speaker ══
load_ollama(SPEAKER_MODEL)
# speaker guard — abort in seconds if the speaker is broken
_chk,_=full_image_question_answer(PILImage.fromarray(masks[0]["npimg"]), QLIST[0])
assert not str(_chk).startswith("(vlm-fail"), f"SPEAKER BROKEN: {_chk}"
print(f"speaker check OK: {str(_chk)[:50]!r}\n")

_run=time.time(); TOTAL=len(BASE_QAMS)*N_IMAGES*len(ESN0_SWEEP); _done=0
for bi,(mb,tag) in enumerate(BASE_QAMS,1):
    if RESUME and os.path.exists(_ckpt_path(tag)):
        print(f"# QAM {bi}/{len(BASE_QAMS)}: {tag} — checkpoint exists, skipping\n", flush=True)
        _done += N_IMAGES*len(ESN0_SWEEP); continue
    print(f"{'#'*56}\n# QAM {bi}/{len(BASE_QAMS)}: {_modname(mb)} ({tag}) — STAR DS={STAR_DS}, conv DS={CONV_DS}\n{'#'*56}")
    tq=time.time()
    records=[]; psnr_store={}
    for ii,mk in enumerate(masks):
        img_pil=PILImage.fromarray(mk["npimg"])
        for e in ESN0_SWEEP:
            with torch.no_grad():
                base=transmit_baseline_x(img_pil,e,mb,CONV_DS)                 # conventional at CONV_DS
                sci,_=transmit_sci_x(img_pil,e,mb,STAR_DS,mk["cache"])         # STAR at STAR_DS
            psnr_store[(tag,e,"base",ii)]=psnr(mk["npimg"],base,mk["crit_pix"])
            psnr_store[(tag,e,"sci", ii)]=psnr(mk["npimg"],sci ,mk["crit_pix"])
            for scheme,recon in [("base",base),("sci",sci)]:
                for q in QLIST:
                    try: ans,_=full_image_question_answer(PILImage.fromarray(recon),q)
                    except Exception as ex: ans=f"(vlm-fail:{str(ex)[:20]})"
                    records.append(dict(qam=tag,mb=mb,snr=e,scheme=scheme,img_idx=ii,
                                        question=q,answer=ans,truth=truths[ii][q]))
            _done+=1; del base,sci; _gpu()
        eta=(TOTAL-_done)/max(_done,1)*(time.time()-_run)/3600
        print(f"  [{tag}] img {ii+1}/{N_IMAGES} ({mk['tmpl']}) | {_mem()} | ~{eta:.1f}h left", flush=True)
    with open(_ckpt_path(tag),"wb") as f:
        pickle.dump({"records":records,"psnr_store":psnr_store,"qlist":QLIST,
                     "config":{"STAR_DS":STAR_DS,"CONV_DS":CONV_DS,"M_CRIT":M_CRIT_RUN,
                               "ESN0_SWEEP":ESN0_SWEEP,"SEED_RUN":SEED_RUN,
                               "BASE_QAMS":BASE_QAMS,"N_IMAGES":N_IMAGES}},f)
    print(f"  [{tag}] ★ ckpt saved | QAM {(time.time()-tq)/60:.0f}min", flush=True)
    records=[]; psnr_store={}; gc.collect()          # RAM flush
    print(f"  [{tag}] flushed RAM | {_mem()}\n", flush=True)
print(f"[Stages 1+2] done in {(time.time()-_run)/3600:.1f}h\n")
unload_ollama(SPEAKER_MODEL)

# ══ STAGE 3: judge — per-QAM from disk ══
load_ollama(JUDGE_MODEL)
t0=time.time()
for _,tag in BASE_QAMS:
    p=_ckpt_path(tag)
    if not os.path.exists(p): print(f"  [{tag}] no ckpt, skip"); continue
    d=pickle.load(open(p,"rb")); recs=d["records"]
    if all("correct" in r for r in recs) and recs:
        print(f"  [{tag}] already judged"); del recs,d; gc.collect(); continue
    for rec in recs:
        if "correct" in rec: continue
        try: ok,_=llm_judge_equivalent(rec["question"],rec["truth"],rec["answer"])
        except Exception as ex: print("  judge fail:", str(ex)[:60]); ok=False
        rec["correct"]=bool(ok)
    pickle.dump(d, open(p,"wb"))
    print(f"  [{tag}] judged {len(recs)} answers", flush=True)
    del recs, d; gc.collect()
print(f"[Stage 3] done in {(time.time()-t0)/3600:.1f}h")
unload_ollama(JUDGE_MODEL)

# ══ STAGE 4: aggregate from checkpoints ══
def _agg_tag(tag):
    d=pickle.load(open(_ckpt_path(tag),"rb")); recs=d["records"]; ps=d["psnr_store"]
    def acc(e,sch):
        v=[r["correct"] for r in recs if r["snr"]==e and r["scheme"]==sch]
        return float(np.mean(v)) if v else 0.0
    def pq(e,sch,q):
        v=[r["correct"] for r in recs if r["snr"]==e and r["scheme"]==sch and r["question"]==q]
        return float(np.mean(v)) if v else 0.0
    def cp(e,sch):
        v=[ps[(tag,e,sch,ii)] for ii in range(N_IMAGES) if (tag,e,sch,ii) in ps]
        return float(np.mean(v)) if v else 0.0
    res=[(e, acc(e,"sci"), acc(e,"base"), cp(e,"sci"), cp(e,"base")) for e in ESN0_SWEEP]
    pq_acc={sch:{e:{q:pq(e,sch,q) for q in QLIST} for e in ESN0_SWEEP} for sch in ("sci","base")}
    del recs, ps, d; gc.collect()
    return {"res":res,"pq_acc":pq_acc}

base_runs={}
for _,tag in BASE_QAMS:
    if os.path.exists(_ckpt_path(tag)):
        base_runs[tag]=_agg_tag(tag)

with open(os.path.join(SAVE_DIR,"base_runs_final.pkl"),"wb") as f:
    pickle.dump(base_runs,f)

order=[("1024qam","1024-QAM","#7C3AED"),("256qam","256-QAM","#DC2626"),("64qam","64-QAM","#EA580C"),
       ("16qam","16-QAM","#16A34A"),("qpsk","QPSK","#2563EB"),("bpsk","BPSK","#0891B2")]
fig,ax=plt.subplots(1,2,figsize=(16,5.5))
for tag,lbl,col in order:
    if tag not in base_runs: continue
    res=base_runs[tag]["res"]; eb=[r[0] for r in res]
    ax[0].plot(eb,[r[1] for r in res],"o-",color=col,lw=2,label=f"STAR ({lbl}, DS={STAR_DS})")
    ax[0].plot(eb,[r[2] for r in res],"s--",color=col,alpha=0.4,label=f"{lbl} (DS={CONV_DS})")
    ax[1].plot(eb,[r[3] for r in res],"o-",color=col,lw=2,label=f"STAR ({lbl}, DS={STAR_DS})")
    ax[1].plot(eb,[r[4] for r in res],"s--",color=col,alpha=0.4,label=f"{lbl} (DS={CONV_DS})")
ax[0].set_xlabel("Es/N0 (dB)");ax[0].set_ylabel("semantic accuracy");ax[0].set_ylim(0,1.02)
ax[0].grid(alpha=0.3);ax[0].legend(fontsize=8,ncol=2);ax[0].set_title("Semantic accuracy")
ax[1].set_xlabel("Es/N0 (dB)");ax[1].set_ylabel("S-PSNR (dB)")
ax[1].grid(alpha=0.3);ax[1].legend(fontsize=8,ncol=2);ax[1].set_title("S-PSNR")
plt.suptitle(f"STAR (DS={STAR_DS}) vs conventional QAM (DS={CONV_DS}) — {N_IMAGES} images",y=1.02,fontsize=13)
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR,"combined_base_qam.png"),dpi=120,bbox_inches="tight"); plt.show()

print(f"\n★★★ ALL DONE → {SAVE_DIR}")

Paper-style plots (IEEE one-column PDF) for one run.

In [ ]:
import os, pickle, numpy as np, matplotlib.pyplot as plt

CONV_DS = 1      # which run to plot (Fig. 4: 4, Fig. 5: 1)
RUN_DIR  = os.path.join(RESULTS_DIR, f"STAR_DS4_conv_DS{CONV_DS}")
SAVE_DIR = FIG_DIR

# load results + the DS pairing from the run's config
base_runs = pickle.load(open(os.path.join(RUN_DIR, "base_runs_final.pkl"), "rb"))
# pull STAR_DS / CONV_DS from any per-QAM checkpoint's config
import glob
_ck = sorted(glob.glob(os.path.join(RUN_DIR, "_ckpt_*.pkl")))
_cfg = pickle.load(open(_ck[0], "rb"))["config"]
STAR_DS = _cfg.get("STAR_DS", 4); CONV_DS = _cfg.get("CONV_DS", 1)
print(f"loaded {RUN_DIR} | STAR DS={STAR_DS}, conv DS={CONV_DS} | tags: {list(base_runs)}")

order = [("1024qam","1024-QAM","#7C3AED"),("256qam","256-QAM","#DC2626"),
         ("64qam","64-QAM","#EA580C"),("16qam","16-QAM","#16A34A"),
         ("qpsk","QPSK","#2563EB"),("bpsk","BPSK","#0891B2")]

INK = "#000000"; GRID_COLOR = "#D8DEE9"
plt.rcParams.update({
    "font.family":"sans-serif",
    "font.sans-serif":["TeX Gyre Heros","Liberation Sans","Helvetica","Arial","DejaVu Sans"],
    "figure.dpi":180,"figure.facecolor":"white","axes.facecolor":"white",
    "text.color":INK,"axes.labelcolor":INK,"xtick.color":INK,"ytick.color":INK,
    "xtick.labelsize":8.5,"ytick.labelsize":8.5,"axes.labelsize":9.5,
    "pdf.fonttype":42,"ps.fonttype":42,
})
def style_axis(ax):
    ax.grid(True, linestyle="--", color=GRID_COLOR, linewidth=0.7, alpha=0.85); ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    for s in ("left","bottom"): ax.spines[s].set_color(INK); ax.spines[s].set_linewidth(1.0)
    ax.tick_params(axis="both", length=4, width=1.0, color=INK)

def make_curve(i_star, i_conv, ylabel, ylim, legend_loc, fname):
    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    for tag, lbl, col in order:
        if tag not in base_runs: continue
        res = base_runs[tag]["res"][1:]          # paper axis starts at -6 dB
        eb = [r[0] for r in res]
        ax.plot(eb, [r[i_star] for r in res], "o-",  color=col, lw=1.5, ms=3.5,
                zorder=3, label=f"STAR ({lbl}, D={STAR_DS})")
        ax.plot(eb, [r[i_conv] for r in res], "s--", color=col, lw=1.2, ms=3.0,
                alpha=0.5, zorder=2, label=f"{lbl} (D={CONV_DS})")
    ax.set_xlabel(r"$E_s/N_0$ (dB)"); ax.set_ylabel(ylabel)
    if ylim: ax.set_ylim(*ylim)
    style_axis(ax)
    leg = ax.legend(fontsize=5.6, ncol=2, loc=legend_loc, framealpha=0.9,
                    handlelength=1.6, handletextpad=0.4, columnspacing=0.8,
                    labelspacing=0.22, borderpad=0.3)
    leg.get_frame().set_edgecolor(GRID_COLOR); leg.get_frame().set_linewidth(0.6)
    fig.tight_layout(pad=0.6)
    out = os.path.join(SAVE_DIR, fname)
    fig.savefig(out, format="pdf", bbox_inches="tight")
    print("Saved:", out); plt.show(); plt.close(fig)

# res tuple = (snr, sci_acc, base_acc, sci_spsnr, base_spsnr)
make_curve(1, 2, "Semantic accuracy", (0, 1.03), "lower right",
           f"STAR_vs_QAM_semantic_accuracy_conv_DS{CONV_DS}.pdf")
make_curve(3, 4, "S-PSNR (dB)", None, "upper left",
           f"STAR_vs_QAM_S-PSNR_conv_DS{CONV_DS}.pdf")
print("Done.")

## 7. Resource elements per image (Fig. 6a)

REs of the robust reference (whole raw image, BPSK + rate-1/3 conv code) vs STAR (256-QAM base at $D=4$
plus the coded protection tier), from Eq. (1). Uses the `masks` of the run in Section 6.

In [ ]:
import os, pickle, numpy as np, matplotlib.pyplot as plt

SAVE_DIR = FIG_DIR
POS_BITS=21; RATE=1/3; CL=7; BASE_QAM=8; STAR_DS=4   # POS_BITS = ceil(log2(1672*941))

def get_image_stats():
    if 'masks' in globals() and masks:
        return [(mk["npimg"].shape[0], mk["npimg"].shape[1], int(mk["crit_pix"].sum())) for mk in masks]
    raise RuntimeError("need `masks` in memory (run Section 6 first)")

stats=get_image_stats()
def base_re(px,m): return int(np.ceil(px*24/m))
def crit_re(n): u=POS_BITS+24; return int(np.ceil(n*(u/RATE+(CL-1)/RATE)))

bpsk_conv=[]; star=[]
for H,W,nc in stats:
    N=H*W
    bpsk_conv.append(int(np.ceil(N*24/RATE)))
    star.append(base_re(N//(STAR_DS*STAR_DS),BASE_QAM)+crit_re(nc))
m_bc=np.mean(bpsk_conv); m_st=np.mean(star); saved=m_bc-m_st

INK = "#000000"; GRID_COLOR = "#D8DEE9"
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["TeX Gyre Heros", "Liberation Sans", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 180, "figure.facecolor": "white", "axes.facecolor": "white",
    "text.color": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "xtick.labelsize": 8.8, "ytick.labelsize": 8.8, "axes.labelsize": 9.8,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

def style_axis(ax):
    ax.grid(axis="y", linestyle="--", color=GRID_COLOR, linewidth=0.8, alpha=0.85)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(INK); ax.spines[s].set_linewidth(1.0)
    ax.tick_params(axis="both", length=4, width=1.0, color=INK)

labels=["BPSK +\nConv-1/3","STAR\n(256-QAM)","Resource elements\nsaved"]
vals=[m_bc, m_st, saved]
colors=["#6B7280","#F5C518","#16A34A"]     # STAR = yellow, saved = green

fig, ax = plt.subplots(figsize=(3.45, 2.4))
x = np.arange(len(vals))
bars = ax.bar(x, np.array(vals)/1e6, color=colors, width=0.62,
              edgecolor=INK, linewidth=0.8, zorder=3)

ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("Resource elements per image\n(in Millions)", linespacing=1.2)
ax.set_ylim(0, max(vals)/1e6 * 1.16)
style_axis(ax)

for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v/1e6 + max(vals)/1e6*0.02, f"{v/1e6:.1f}M",
            ha="center", va="bottom", fontsize=8.6, fontweight="bold", color=INK, clip_on=False)

fig.tight_layout(pad=0.7)
fig.subplots_adjust(left=0.17)   # keeps the two-line y-label from clipping
out = os.path.join(SAVE_DIR, "STAR_resource_saving_BPSKconv_vs_STAR256_DS4.pdf")
fig.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out); plt.show(); plt.close(fig)

print(f"BPSK+Conv-1/3 : {m_bc/1e6:.2f}M REs")
print(f"STAR (256-QAM): {m_st/1e6:.2f}M REs")
print(f"REs saved      : {saved/1e6:.2f}M  ({saved:,.0f} REs per image)")

## 8. Selective criticality (Fig. 6b)

Only the room task (or room + fire) is critical. Four variants: 256-QAM at $D=1$ and $D=4$, STAR with the
room task critical, STAR with fire + room critical.

In [ ]:
import random, os, pickle, numpy as np, matplotlib.pyplot as plt

SAVE_DIR=os.path.join(RESULTS_DIR, "selective_criticality")
os.makedirs(SAVE_DIR,exist_ok=True); print("SAVE_DIR:",SAVE_DIR)

Q_ROOM="Which room is this?"; Q_FIRE="Is there a fire?"
QSET=[Q_FIRE,Q_ROOM]; QID_ROOM=qid_of(Q_ROOM); QID_FIRE=qid_of(Q_FIRE)
M_BASE_EXP=8; ESN0_SWEEP=[-8,-6,-4,-2,0,1,2,4,8,12,16]
N_PER_TEMPLATE=20; SEED_RUN=20240612
POS_BITS=21; RATE=CONV_RATE; CL=CONV_CL
CONFIGS={"256-QAM DS=1":"#93C5FD","256-QAM DS=4":"#F59E0B",
         "STAR (room critical)":"#F5C518","STAR (fire+room critical)":"#DC2626"}

rng=random.Random(SEED_RUN); test_imgs=[]
for tmpl in ["fire","human","clear"]:
    cands=[p for p in panel_records if p["meta"]["template"]==tmpl]
    test_imgs+=rng.sample(cands,min(N_PER_TEMPLATE,len(cands)))
rng.shuffle(test_imgs); N_IMG=len(test_imgs)
print(f"{N_IMG} imgs | questions {QSET}\n")

# ══ PHASE 0: HF attention → TWO masks per image (room-only, fire+room) → free HF ══
print("[Phase 0] HF Qwen2.5-VL → masks (room-only + fire+room)")
load_hf_model()
def build_mask(img, critset):
    T,_,_,_=BuildSCI(img,critset,tau=TAU,smooth=1.2,dilate=3,sink_drop=4,token_pool="max")
    crit_pix=(T[...,3]==2); npimg=np.array(img,np.uint8); N=npimg.shape[0]*npimg.shape[1]
    ci=np.where(crit_pix.reshape(-1))[0]; crgb=npimg.reshape(-1,3)[ci]; POS=int(np.ceil(np.log2(N)))
    return dict(crit_pix=crit_pix, cache=(ci,crgb,POS,crit_pix), n_crit=len(ci))

masks=[]
for panel in test_imgs:
    img=panel["img"].convert("RGB"); npimg=np.array(img,np.uint8)
    m_room = build_mask(img, {QID_ROOM:2})            # room only
    m_both = build_mask(img, {QID_ROOM:2, QID_FIRE:2})# fire + room
    masks.append(dict(npimg=npimg, tmpl=panel["meta"]["template"], room=m_room, both=m_both))
    print(f"  {panel['meta']['template']}: room-crit={100*m_room['crit_pix'].mean():.1f}% | "
          f"fire+room-crit={100*m_both['crit_pix'].mean():.1f}%")
free_hf_model()

# ══ PHASE 1: transmit ALL 4 variants → store recons ══
print("\n[Phase 1] transmission (no LLM)")
recons={}
for ii,(panel,mk) in enumerate(zip(test_imgs,masks)):
    img=panel["img"].convert("RGB")
    for e in ESN0_SWEEP:
        recons[(ii,e,"256-QAM DS=1")]        = transmit_baseline_x(img,e,M_BASE_EXP,1)
        recons[(ii,e,"256-QAM DS=4")]        = transmit_baseline_x(img,e,M_BASE_EXP,4)
        recons[(ii,e,"STAR (room critical)")]= transmit_sci_x(img,e,M_BASE_EXP,4,mk["room"]["cache"])[0]
        recons[(ii,e,"STAR (fire+room critical)")]= transmit_sci_x(img,e,M_BASE_EXP,4,mk["both"]["cache"])[0]
    gc.collect(); torch.cuda.empty_cache()
    print(f"  transmit img {ii+1}/{N_IMG} ({mk['tmpl']})", flush=True)
print(f"  {len(recons)} reconstructions stored")

# ══ PHASE 2: speaker VLM → answers ══
print("\n[Phase 2] speaker VLM → answers")
load_ollama(SPEAKER_MODEL)
answers={}
for (ii,e,cfg),recon in recons.items():
    for q in QSET:
        try: a,_=full_image_question_answer(PILImage.fromarray(recon),q)
        except Exception as ex: a=f"(fail:{str(ex)[:15]})"
        answers[(ii,e,cfg,q)]=a
print(f"  {len(answers)} answers")
unload_ollama(SPEAKER_MODEL)
del recons; gc.collect(); torch.cuda.empty_cache()

# ══ PHASE 3: judge → grade ══
print("\n[Phase 3] judge → grade")
load_ollama(JUDGE_MODEL)
records=[]
for ii,(panel,mk) in enumerate(zip(test_imgs,masks)):
    for e in ESN0_SWEEP:
        for cfg in CONFIGS:
            for q in QSET:
                ans=answers[(ii,e,cfg,q)]; gt=answer_for_question(panel["meta"],q)
                try: ok,_=llm_judge_equivalent(q,gt,ans)
                except Exception as ex: print("  judge fail",str(ex)[:30]); ok=False
                records.append(dict(config=cfg,img_idx=ii,template=mk["tmpl"],snr=e,
                                    question=q,answer=ans,truth=gt,correct=bool(ok)))
    print(f"  judged img {ii+1}/{N_IMG}", flush=True)
unload_ollama(JUDGE_MODEL)

# ══ PHASE 4: aggregate + RE + save + plots ══
print("\n[Phase 4] aggregate + save")
def acc(cfg,q,e):
    v=[r["correct"] for r in records if r["config"]==cfg and r["question"]==q and r["snr"]==e]
    return float(np.mean(v)) if v else 0.0
acc_mean={c:{q:{e:acc(c,q,e) for e in ESN0_SWEEP} for q in QSET} for c in CONFIGS}

def base_re(px,m): return int(np.ceil(px*24/m))
def crit_re(n): u=POS_BITS+24; return int(np.ceil(n*(u/RATE+(CL-1)/RATE)))
re_vals={c:[] for c in CONFIGS}
for mk in masks:
    N=mk["npimg"].shape[0]*mk["npimg"].shape[1]
    re_vals["256-QAM DS=1"].append(base_re(N,M_BASE_EXP))
    re_vals["256-QAM DS=4"].append(base_re(N//16,M_BASE_EXP))
    re_vals["STAR (room critical)"].append(base_re(N//16,M_BASE_EXP)+crit_re(mk["room"]["n_crit"]))
    re_vals["STAR (fire+room critical)"].append(base_re(N//16,M_BASE_EXP)+crit_re(mk["both"]["n_crit"]))
rm={k:float(np.mean(v)) for k,v in re_vals.items()}

stamp=time.strftime("%Y%m%d_%H%M%S")
fname=os.path.join(SAVE_DIR,f"room_experiment_4var_{stamp}.pkl")
with open(fname,"wb") as f:
    pickle.dump({"records":records,"acc_mean":acc_mean,"re_vals":re_vals,"re_mean":rm,
                 "QSET":QSET,"ESN0_SWEEP":ESN0_SWEEP,"CONFIGS":list(CONFIGS.keys()),
                 "config":{"M_BASE":M_BASE_EXP,"RATE":RATE,"CL":CL,"POS_BITS":POS_BITS,
                           "SEED_RUN":SEED_RUN,"N_IMAGES":N_IMG},
                 "images":[m["tmpl"] for m in masks]},f)
print(f"★ saved {fname} ({len(records)} records)")

# accuracy plots (per question)
fig,ax=plt.subplots(1,2,figsize=(14,5))
for qi,q in enumerate(QSET):
    for cfg,col in CONFIGS.items():
        ax[qi].plot(ESN0_SWEEP,[acc_mean[cfg][q][e] for e in ESN0_SWEEP],"o-",color=col,lw=2,label=cfg)
    ax[qi].set_title(f'"{q}"');ax[qi].set_xlabel("Es/N0 (dB)");ax[qi].set_ylabel("semantic accuracy")
    ax[qi].set_ylim(0,1.03);ax[qi].grid(alpha=0.3);ax[qi].legend(fontsize=8)
plt.suptitle("Selective criticality — 4 variants",y=1.02,fontsize=13)
plt.tight_layout(); plt.show()

# RE bar
fig,axb=plt.subplots(figsize=(8,5))
bars=axb.bar(range(4),[rm[k]/1e6 for k in CONFIGS],color=[CONFIGS[k] for k in CONFIGS],edgecolor="black")
axb.set_xticks(range(4));axb.set_xticklabels(list(CONFIGS),fontsize=8,rotation=12)
axb.set_ylabel("Resource Elements per image (millions)");axb.set_title("Resources used");axb.grid(alpha=0.3,axis="y")
for b,k in zip(bars,CONFIGS): axb.text(b.get_x()+b.get_width()/2,rm[k]/1e6*1.02,f"{rm[k]/1e6:.2f}M",ha="center",fontsize=8,fontweight="bold")
plt.tight_layout(); plt.show()
for k in CONFIGS: print(f"{k:26s}: {rm[k]/1e6:.3f}M REs")

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
from matplotlib.patches import Patch

ESN0_SHOW = 2   # SNR point to show (must be in ESN0_SWEEP)
assert ESN0_SHOW in ESN0_SWEEP, f"{ESN0_SHOW} not in {ESN0_SWEEP}"

SAVE_DIR = FIG_DIR

INK="#000000"; GRID_COLOR="#D8DEE9"
plt.rcParams.update({
    "font.family":"sans-serif",
    "font.sans-serif":["TeX Gyre Heros","Liberation Sans","Helvetica","Arial","DejaVu Sans"],
    "figure.dpi":180,"figure.facecolor":"white","axes.facecolor":"white",
    "text.color":INK,"axes.labelcolor":INK,"xtick.color":INK,"ytick.color":INK,
    "xtick.labelsize":8.0,"ytick.labelsize":8.5,"axes.labelsize":9.5,
    "pdf.fonttype":42,"ps.fonttype":42,
})
def style_axis(ax):
    ax.grid(axis="y",linestyle="--",color=GRID_COLOR,linewidth=0.8,alpha=0.85); ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    for s in ("left","bottom"): ax.spines[s].set_color(INK); ax.spines[s].set_linewidth(1.0)
    ax.tick_params(axis="both",length=4,width=1.0,color=INK)

SHOW_CONFIGS = ["256-QAM DS=4", "256-QAM DS=1", "STAR (room critical)"]
cfg_color = {
    "256-QAM DS=1": "#2563EB",           # blue
    "256-QAM DS=4": "#7C3AED",           # purple
    "STAR (room critical)": "#F5C518",   # yellow
}
label_map = {
    "256-QAM DS=4": "256-QAM\nDS=4",
    "256-QAM DS=1": "256-QAM\nDS=1",
    "STAR (room critical)": "STAR\n(256-QAM, DS=4)",
}

x = np.arange(len(SHOW_CONFIGS)); width = 0.38
fig, ax = plt.subplots(figsize=(3.45, 2.5))

for i, cfg in enumerate(SHOW_CONFIGS):
    col = cfg_color[cfg]
    v_fire = acc_mean[cfg][Q_FIRE][ESN0_SHOW]
    v_room = acc_mean[cfg][Q_ROOM][ESN0_SHOW]
    ax.bar(x[i]-width/2, v_fire, width, color=col, edgecolor=INK, linewidth=0.7, zorder=3)
    ax.bar(x[i]+width/2, v_room, width, color=col, edgecolor=INK, linewidth=0.7,
           alpha=0.45, hatch="//", zorder=3)
    ax.text(x[i]-width/2, v_fire+0.02, f"{v_fire:.2f}", ha="center", va="bottom",
            fontsize=7.5, fontweight="bold", color=INK, clip_on=False)
    ax.text(x[i]+width/2, v_room+0.02, f"{v_room:.2f}", ha="center", va="bottom",
            fontsize=7.5, fontweight="bold", color=INK, clip_on=False)

ax.set_xticks(x); ax.set_xticklabels([label_map[c] for c in SHOW_CONFIGS])
ax.set_ylabel("Semantic accuracy"); ax.set_ylim(0, 1.20)
style_axis(ax)

ax.legend(handles=[
    Patch(facecolor="gray", edgecolor=INK, label="Fire"),
    Patch(facecolor="gray", edgecolor=INK, alpha=0.45, hatch="//", label="Room"),
], fontsize=7.2, loc="upper center", ncol=2, framealpha=0.95,
   handlelength=1.5, handletextpad=0.4, columnspacing=1.0, borderpad=0.35,
   bbox_to_anchor=(0.5, 1.16)).get_frame().set_edgecolor(GRID_COLOR)

fig.tight_layout(pad=0.7)
out = os.path.join(SAVE_DIR, f"STAR_selective_criticality_accuracy_{ESN0_SHOW}dB.pdf")
fig.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out); plt.show(); plt.close(fig)

## 9. SCI overlays (Fig. 2) and received images (Fig. 3)

Fig. 2: continuous SCI map on one fire image for four critical-task sets.
Fig. 3: received images with a 256-QAM base at $E_s/N_0 = 2$ dB.

In [ ]:
import os, random, numpy as np, matplotlib.pyplot as plt
from PIL import Image as PILImage

SAVE_DIR = FIG_DIR
load_hf_model()

Q_FIRE = "Is there a fire?"
Q_ROOM = "Which room is this?"
QID_FIRE = qid_of(Q_FIRE)
QID_ROOM = qid_of(Q_ROOM)
ALL_QIDS = list(range(len(ACTIVE_QUESTION_TYPES)))
TUNE = dict(smooth=1.2, dilate=3, sink_drop=4, token_pool="max")

# critical set (qid: score) → output filename suffix
PANELS = [
    ("fire",       {QID_FIRE: 2}),
    ("room",       {QID_ROOM: 2}),
    ("fire+room",  {QID_FIRE: 2, QID_ROOM: 2}),
    ("all",        {q: 2 for q in ALL_QIDS}),
]

# one fire-template panel (fixed seed)
SEED = 2100186420
rng = random.Random(SEED)
img = rng.choice([p for p in panel_records if p["meta"]["template"] == "fire"])["img"].convert("RGB")
npimg = np.array(img, np.uint8); H, W = npimg.shape[:2]
print(f"[selective-protection] SEED={SEED}  fire panel  {W}x{H}")

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["TeX Gyre Heros", "Liberation Sans", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 180, "figure.facecolor": "white", "pdf.fonttype": 42, "ps.fonttype": 42,
})

def combined_map(critset):
    combined, _ = importance_map(img, critset, combine="max", **TUNE)
    return np.array(PILImage.fromarray((combined*255).astype(np.uint8))
                    .resize((W, H), PILImage.BILINEAR)) / 255.0

def save_overlay(critset, suffix):
    hu = combined_map(critset)
    # size the figure to the image aspect ratio so there's no whitespace
    fig_w = 2.2
    fig_h = fig_w * H / W
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.imshow(npimg)
    ax.imshow(hu, cmap="jet", alpha=0.5)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(True); s.set_color("#000000"); s.set_linewidth(0.8)
    ax.margins(0)
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    out = os.path.join(SAVE_DIR, f"STAR_selective_protection_fire_{suffix}.pdf")
    fig.savefig(out, format="pdf", bbox_inches="tight", pad_inches=0.02)
    print("Saved:", out); plt.show(); plt.close(fig)

for suffix, critset in PANELS:
    save_overlay(critset, suffix)

print("Done.")

In [ ]:
import os, numpy as np, torch, matplotlib.pyplot as plt
from PIL import Image as PILImage

SAVE_DIR = FIG_DIR
load_hf_model()

M_BASE_SHOW = 8      # 256-QAM base
ESN0_SHOW   = 2
SEED        = 1234   # fixes the channel noise realisation for reproducibility

Q_ROOM = "Which room is this?"
QID_ROOM = qid_of(Q_ROOM)
_TAU = TAU

# ── pick the PUMP ROOM panel (prefer fire template so All-Q adds the flame) ──
def room_of(panel):
    return str(answer_for_question(panel["meta"], Q_ROOM))

pump = [p for p in panel_records if "PUMP" in room_of(p).upper()]
if not pump:
    rooms = sorted({room_of(p) for p in panel_records})
    raise RuntimeError(f"No PUMP ROOM panel found. Available rooms: {rooms}")
pump_fire = [p for p in pump if p["meta"]["template"] == "fire"]
panel = (pump_fire or pump)[0]

img = panel["img"].convert("RGB"); npimg = np.array(img, np.uint8); H, W = npimg.shape[:2]
print(f"selected: {panel['meta']['template']} | room={room_of(panel)} | {W}x{H}")

def psnr(a, b, mask=None):
    a = a.astype(np.float64); b = b.astype(np.float64)
    if mask is not None: a = a[mask]; b = b[mask]
    e = np.mean((a - b)**2); return 99 if e == 0 else 10*np.log10(255**2/e)

def build_cache(critset):
    T = BuildSCI(img, critset, tau=_TAU, smooth=1.2, dilate=3, sink_drop=4, token_pool="max")[0]
    crit_pix = (T[..., 3] == 2); N = H*W
    ci = np.where(crit_pix.reshape(-1))[0]; crgb = npimg.reshape(-1, 3)[ci]
    POS = int(np.ceil(np.log2(N)))
    return (ci, crgb, POS, crit_pix)

# masks: room-only, and all-questions
cache_room = build_cache({QID_ROOM: 2})
_, crit_pix_all, ci_all, crgb_all, POS_all = _sci_mask_allq(img)
cache_all = (ci_all, crgb_all, POS_all, crit_pix_all)
print(f"crit fraction | room={100*cache_room[3].mean():.1f}%  all-Q={100*crit_pix_all.mean():.1f}%")

# ── reconstruct all four (fixed seed → same channel draw) ──────────────────
np.random.seed(SEED); torch.manual_seed(SEED)
recon = {
    "conv_256QAM_DS4":            transmit_baseline_x(img, ESN0_SHOW, M_BASE_SHOW, 4),
    "conv_256QAM_DS1":            transmit_baseline_x(img, ESN0_SHOW, M_BASE_SHOW, 1),
    "STAR_256QAM_DS4_critRoom":   transmit_sci_x(img, ESN0_SHOW, M_BASE_SHOW, 4, cache_room)[0],
    "STAR_256QAM_DS4_critAllQ":   transmit_sci_x(img, ESN0_SHOW, M_BASE_SHOW, 4, cache_all)[0],
}
# PSNR on the room region (common reference across configs)
rmask = cache_room[3]
for k, arr in recon.items():
    print(f"  {k:28s} room-PSNR={psnr(npimg, arr, rmask):.1f} dB")

# ── save each as its own PDF ───────────────────────────────────────────────
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["TeX Gyre Heros", "Liberation Sans", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 180, "figure.facecolor": "white", "pdf.fonttype": 42, "ps.fonttype": 42,
})

def save_img(arr, suffix):
    fig_w = 3.4; fig_h = fig_w * H / W
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.imshow(arr); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(True); s.set_color("#000000"); s.set_linewidth(0.8)
    ax.margins(0); fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    out = os.path.join(SAVE_DIR, f"STAR_recon_{ESN0_SHOW}dB_pumproom_{suffix}.pdf")
    fig.savefig(out, format="pdf", bbox_inches="tight", pad_inches=0.02, dpi=300)
    print("Saved:", out); plt.show(); plt.close(fig)

for suffix, arr in recon.items():
    save_img(arr, suffix)
print("Done.")

## 10. Computation time of SCI estimation (Sec. IV-C)

In [ ]:
import time, random, numpy as np, torch
load_hf_model()

N_TIME = 10                                   # images to time
TUNE = dict(smooth=1.2, dilate=3, sink_drop=4, token_pool="max")
q0 = ACTIVE_QUESTION_TYPES[0]
crit1 = {ACTIVE_QUESTION_TYPES.index(q0): 2}  # one task

imgs = [p["img"].convert("RGB") for p in random.Random(0).sample(panel_records, N_TIME + 1)]

_ = importance_map(imgs[0], crit1, combine="max", **TUNE)   # warm-up, not timed

times = []
for img in imgs[1:]:
    torch.cuda.synchronize(); t0 = time.perf_counter()
    _ = importance_map(img, crit1, combine="max", **TUNE)
    torch.cuda.synchronize(); times.append(time.perf_counter() - t0)

t = np.array(times)
print(f"task: {q0!r}")
print(f"image size: {imgs[1].size}")
print(f"SCI time per task and image: mean={t.mean():.2f} s | std={t.std():.2f} s | min={t.min():.2f} | max={t.max():.2f}  (n={len(t)})")